In [2]:
import pandas as pd
import numpy as np
import cv2
import os

In [3]:
test_path = "/Users/shanks/Downloads/test_set"
train_path = "/Users/shanks/Downloads/training_set"

In [19]:
IMG_SIZE = 32

def load_data(path):
    X = []
    Y = []

    labels = [l for l in os.listdir(path) if os.path.isdir(os.path.join(path, l))]

    for label_index, label in enumerate(labels):
        folder_path = os.path.join(path, label)

        for file in os.listdir(folder_path):
            img_path = os.path.join(folder_path, file)

            if not os.path.isfile(img_path):
                continue

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            X.append(img)
            Y.append(label_index)

    return np.array(X), np.array(Y), labels

In [20]:
train_path = "/Users/shanks/Downloads/training_set"
test_path = "/Users/shanks/Downloads/test_set"

X_train, Y_train, labels = load_data(train_path)
X_test, Y_test, _ = load_data(test_path)

In [21]:
X_train = X_train / 255.0
X_test = X_test / 255.0

In [22]:
X_train = X_train.reshape(X_train.shape[0], -1).T
X_test = X_test.reshape(X_test.shape[0], -1).T

In [23]:
Y_train = Y_train.reshape(-1).astype(int)
Y_test = Y_test.reshape(-1).astype(int)

In [24]:
Y_train = Y_train.reshape(1, -1)
Y_test = Y_test.reshape(1, -1)

In [28]:
input_size = 3072
hidden1 = 512
hidden2 = 256
output_size = 1

W1 = np.random.randn(hidden1, input_size) * 0.01
b1 = np.zeros((hidden1,1))

W2 = np.random.randn(hidden2, hidden1) * 0.01
b2 = np.zeros((hidden2,1))

W3 = np.random.randn(output_size, hidden2) * 0.01
b3 = np.zeros((output_size,1))

In [29]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [34]:
lr = 0.005
epochs = 2000

for epoch in range(epochs):

    Z1 = np.dot(W1, X_train) + b1
    A1 = relu(Z1)

    Z2 = np.dot(W2, A1) + b2
    A2 = relu(Z2)

    Z3 = np.dot(W3, A2) + b3
    A3 = sigmoid(Z3)

    m = Y_train.shape[1]
    loss = - (1/m) * np.sum(
        Y_train * np.log(A3 + 1e-8) +
        (1 - Y_train) * np.log(1 - A3 + 1e-8)
    )

    dZ3 = A3 - Y_train
    dW3 = np.dot(dZ3, A2.T) / m
    db3 = np.sum(dZ3, axis=1, keepdims=True) / m

    dA2 = np.dot(W3.T, dZ3)
    dZ2 = dA2 * (A2 > 0)

    dW2 = np.dot(dZ2, A1.T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * (A1 > 0)

    dW1 = np.dot(dZ1, X_train.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2
    W3 -= lr * dW3
    b3 -= lr * db3

    if epoch % 100 == 0:
        print("Epoch:", epoch, "Loss:", loss)

Epoch: 0 Loss: 0.693073487019858
Epoch: 100 Loss: 0.6930330113209421
Epoch: 200 Loss: 0.6929921772109255
Epoch: 300 Loss: 0.6929510967374737
Epoch: 400 Loss: 0.6929094687355536
Epoch: 500 Loss: 0.6928670862415728
Epoch: 600 Loss: 0.6928243210622591
Epoch: 700 Loss: 0.6927807014074342
Epoch: 800 Loss: 0.6927362167993171
Epoch: 900 Loss: 0.692690862113147
Epoch: 1000 Loss: 0.6926442886649121
Epoch: 1100 Loss: 0.692595818531917
Epoch: 1200 Loss: 0.6925452596307832
Epoch: 1300 Loss: 0.6924926537969881
Epoch: 1400 Loss: 0.6924372962970398
Epoch: 1500 Loss: 0.6923788445638035
Epoch: 1600 Loss: 0.6923169657172864
Epoch: 1700 Loss: 0.6922515709014141
Epoch: 1800 Loss: 0.6921817535190359
Epoch: 1900 Loss: 0.692107798679792


In [35]:
Z1 = np.dot(W1, X_test) + b1
A1 = relu(Z1)

Z2 = np.dot(W2, A1) + b2
A2 = relu(Z2)

Z3 = np.dot(W3, A2) + b3
A3_test = sigmoid(Z3)

predictions = (A3_test > 0.5).astype(int)
accuracy = np.mean(predictions == Y_test)
print("Test Accuracy:", accuracy * 100, "%")

Test Accuracy: 55.80820563519525 %
